# Обзор refined full-covariance HMM grid

Ноутбук читает результаты `train_hmm_refined_full_grid.py`: модели `covariance_type=full`, `n_components=2..10`, top random states из первого broad grid. Фокус: BIC/AIC/LogLik, occupancy, полные распределения длительностей, transition matrix, state means и state covariances.

In [ ]:
from __future__ import annotations

import io
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "build_price_feature_day.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from build_price_feature_day import make_s3_client

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 200)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
S3_BUCKET = "binance-data-downloader"
OUTPUT_PREFIX = "features/hmm_grid_search/refined_full_results"

# Пример: RUN_ID = "refined_full_20260705_120000". None возьмёт самый свежий run.
RUN_ID = None
EXPECTED_MODELS = 135

s3 = make_s3_client()

In [ ]:
def list_s3_keys(bucket: str, prefix: str) -> list[str]:
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix.rstrip("/") + "/"):
        keys.extend(item["Key"] for item in page.get("Contents", []))
    return keys


def s3_key_exists(bucket: str, key: str) -> bool:
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except ClientError as exc:
        code = exc.response.get("Error", {}).get("Code")
        if code in {"404", "NoSuchKey", "NotFound"}:
            return False
        raise


def read_s3_parquet(bucket: str, key: str) -> pd.DataFrame:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    return pd.read_parquet(io.BytesIO(body))


def all_results_key(run_id: str) -> str:
    return f"{OUTPUT_PREFIX.strip('/')}/runs/run_id={run_id}/all_results.parquet"


def models_prefix(run_id: str) -> str:
    return f"{OUTPUT_PREFIX.strip('/')}/runs/run_id={run_id}/models"


def discover_run_ids() -> pd.DataFrame:
    keys = list_s3_keys(S3_BUCKET, f"{OUTPUT_PREFIX.strip('/')}/runs")
    rows = []
    for key in keys:
        if "/run_id=" not in key:
            continue
        run_id = key.split("/run_id=", 1)[1].split("/", 1)[0]
        rows.append({"run_id": run_id, "key": key, "is_all_results": key.endswith("/all_results.parquet"), "is_model_result": key.endswith("/result.parquet")})
    if not rows:
        return pd.DataFrame(columns=["run_id", "has_all_results", "model_files"])
    frame = pd.DataFrame(rows)
    return (
        frame.groupby("run_id", as_index=False)
        .agg(has_all_results=("is_all_results", "any"), model_files=("is_model_result", "sum"))
        .sort_values("run_id", ascending=False)
        .reset_index(drop=True)
    )


def load_results(run_id: str | None) -> tuple[str, pd.DataFrame]:
    runs = discover_run_ids()
    display(runs.head(20))
    if run_id is None:
        candidates = runs[runs["has_all_results"]]
        if candidates.empty:
            candidates = runs[runs["model_files"].gt(0)]
        if candidates.empty:
            raise FileNotFoundError(f"No refined result runs found under s3://{S3_BUCKET}/{OUTPUT_PREFIX}/runs/")
        run_id = candidates.iloc[0]["run_id"]

    final_key = all_results_key(run_id)
    if s3_key_exists(S3_BUCKET, final_key):
        return run_id, read_s3_parquet(S3_BUCKET, final_key)

    result_keys = [key for key in list_s3_keys(S3_BUCKET, models_prefix(run_id)) if key.endswith("/result.parquet")]
    if not result_keys:
        raise FileNotFoundError(f"No refined model result files found for run_id={run_id}")
    frames = [read_s3_parquet(S3_BUCKET, key) for key in sorted(result_keys)]
    return run_id, pd.concat(frames, ignore_index=True)

In [ ]:
run_id, raw_results = load_results(RUN_ID)
print(f"Loaded run_id={run_id}")
print(f"Rows: {len(raw_results):,}")
raw_results.head()

In [ ]:
results = raw_results.copy()
numeric_columns = ["n_components", "random_state", "n_rows", "train_seconds", "n_iter", "log_likelihood", "aic", "bic", "selected_from_broad_bic", "selected_from_broad_n_components"]
for column in numeric_columns:
    if column in results.columns:
        results[column] = pd.to_numeric(results[column], errors="coerce")
results["success"] = results["success"].astype(bool)
results["converged"] = results["converged"].astype("boolean")

successful = results[results["success"]].copy()
failed = results[~results["success"]].copy()

print(f"Expected: {EXPECTED_MODELS}")
print(f"Actual: {len(results)}")
print(f"Success: {len(successful)}")
print(f"Failed: {len(failed)}")
print(f"Missing vs expected: {EXPECTED_MODELS - len(results)}")

## Лучшие модели по BIC/AIC/LogLik

In [ ]:
metric_columns = ["model_group", "n_components", "covariance_type", "random_state", "bic", "aic", "log_likelihood", "converged", "n_iter", "train_seconds", "selected_from_broad_bic", "selected_from_broad_n_components"]

best_by_bic = successful.loc[successful.groupby("model_group")["bic"].idxmin(), metric_columns].sort_values("model_group")
best_by_aic = successful.loc[successful.groupby("model_group")["aic"].idxmin(), metric_columns].sort_values("model_group")
best_by_ll = successful.loc[successful.groupby("model_group")["log_likelihood"].idxmax(), metric_columns].sort_values("model_group")

best_by_bic

In [ ]:
print("Best by AIC")
display(best_by_aic)
print("Best by LogLik")
display(best_by_ll)

In [ ]:
hyper_summary = (
    successful.groupby(["model_group", "n_components"], as_index=False)
    .agg(
        models=("random_state", "count"),
        bic_min=("bic", "min"),
        bic_median=("bic", "median"),
        bic_std=("bic", "std"),
        aic_min=("aic", "min"),
        ll_max=("log_likelihood", "max"),
        converged_rate=("converged", "mean"),
        median_seconds=("train_seconds", "median"),
    )
    .sort_values(["model_group", "bic_min"])
)
hyper_summary

In [ ]:
groups = [group for group in ["price", "flow", "market"] if group in successful["model_group"].unique()]
fig, axes = plt.subplots(1, len(groups), figsize=(6 * len(groups), 4.5), constrained_layout=True, squeeze=False)

for axis, group in zip(axes[0], groups):
    frame = hyper_summary[hyper_summary["model_group"].eq(group)].sort_values("n_components")
    axis.plot(frame["n_components"], frame["bic_min"], marker="o", linewidth=2, label="min BIC")
    axis.plot(frame["n_components"], frame["bic_median"], marker="s", linewidth=1.5, label="median BIC")
    axis.set_title(f"{group}: BIC vs n_components")
    axis.set_xlabel("n_components")
    axis.set_ylabel("BIC lower is better")
    axis.legend()

plt.show()

## JSON parsers

In [ ]:
def parse_json(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, str):
        return json.loads(value)
    return value


def parse_array(value) -> np.ndarray | None:
    parsed = parse_json(value)
    if parsed is None:
        return None
    return np.asarray(parsed, dtype="float64")


def selected_best_rows(metric: str = "bic") -> pd.DataFrame:
    if metric in {"bic", "aic"}:
        return successful.loc[successful.groupby("model_group")[metric].idxmin()].sort_values("model_group")
    return successful.loc[successful.groupby("model_group")[metric].idxmax()].sort_values("model_group")


selected_best = selected_best_rows("bic")
selected_best.loc[:, metric_columns]

## Occupancy и средние длительности для лучших BIC

In [ ]:
duration_rows = []
for _, row in selected_best.iterrows():
    occupancy = parse_array(row["state_occupancy"])
    stats_by_state = parse_json(row["state_duration_stats"])
    for state_id, state_stats in enumerate(stats_by_state):
        duration_rows.append(
            {
                "model_group": row["model_group"],
                "n_components": int(row["n_components"]),
                "random_state": int(row["random_state"]),
                "state": state_id,
                "occupancy": occupancy[state_id],
                **state_stats,
            }
        )

duration_stats_table = pd.DataFrame(duration_rows)
duration_stats_table

In [ ]:
fig, axes = plt.subplots(len(groups), 2, figsize=(15, 4.5 * len(groups)), constrained_layout=True, squeeze=False)

for row_idx, group in enumerate(groups):
    frame = duration_stats_table[duration_stats_table["model_group"].eq(group)]
    axes[row_idx][0].bar(frame["state"], frame["occupancy"], color="#4C78A8")
    axes[row_idx][0].set_title(f"{group}: occupancy")
    axes[row_idx][0].set_xlabel("State")
    axes[row_idx][0].set_ylabel("Share")

    axes[row_idx][1].bar(frame["state"], frame["mean"], color="#F58518")
    axes[row_idx][1].set_title(f"{group}: mean duration")
    axes[row_idx][1].set_xlabel("State")
    axes[row_idx][1].set_ylabel("Minutes")

plt.show()

## Полное распределение длительностей режимов

In [ ]:
def plot_duration_distribution(row: pd.Series, max_duration_quantile: float = 0.99) -> None:
    durations_by_state = parse_json(row["state_duration_distribution"])
    n_states = len(durations_by_state)
    ncols = min(3, n_states)
    nrows = int(np.ceil(n_states / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), constrained_layout=True, squeeze=False)
    fig.suptitle(
        f"{row['model_group']} best BIC: n={int(row['n_components'])}, seed={int(row['random_state'])}",
        fontsize=14,
    )

    for state_id, durations in enumerate(durations_by_state):
        axis = axes[state_id // ncols][state_id % ncols]
        values = np.asarray(durations, dtype="float64")
        if len(values) == 0:
            axis.set_title(f"State {state_id}: empty")
            continue
        upper = max(1, np.quantile(values, max_duration_quantile))
        clipped = values[values <= upper]
        axis.hist(clipped, bins=min(60, max(10, int(np.sqrt(len(clipped))))), color="#4C78A8", alpha=0.8)
        axis.axvline(values.mean(), color="#E45756", linewidth=2, label=f"mean={values.mean():.1f}")
        axis.set_title(f"State {state_id}: count={len(values)}, max={values.max():.0f}")
        axis.set_xlabel("Duration, minutes")
        axis.set_ylabel("Runs")
        axis.legend()

    for state_id in range(n_states, nrows * ncols):
        axes[state_id // ncols][state_id % ncols].set_axis_off()
    plt.show()


for _, row in selected_best.iterrows():
    plot_duration_distribution(row)

## Transition matrices

In [ ]:
fig, axes = plt.subplots(1, len(groups), figsize=(5.8 * len(groups), 5), constrained_layout=True, squeeze=False)

for axis, (_, row) in zip(axes[0], selected_best.iterrows()):
    transition_matrix = parse_array(row["transition_matrix"])
    image = axis.imshow(transition_matrix, vmin=0, vmax=1, cmap="Blues")
    axis.set_title(f"{row['model_group']}: transition matrix")
    axis.set_xlabel("To state")
    axis.set_ylabel("From state")
    ticks = np.arange(transition_matrix.shape[0])
    axis.set_xticks(ticks)
    axis.set_yticks(ticks)
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)

plt.show()

## State means и covariance matrices

In [ ]:
state_mean_rows = []
for _, row in selected_best.iterrows():
    features = parse_json(row["features"])
    means = parse_array(row["state_means"])
    for state_id in range(means.shape[0]):
        state_mean_rows.append({"model_group": row["model_group"], "state": state_id, **dict(zip(features, means[state_id]))})

state_means_table = pd.DataFrame(state_mean_rows)
state_means_table

In [ ]:
for _, row in selected_best.iterrows():
    features = parse_json(row["features"])
    covariances = parse_array(row["state_covariances"])
    n_states = covariances.shape[0]
    ncols = min(3, n_states)
    nrows = int(np.ceil(n_states / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 4.2 * nrows), constrained_layout=True, squeeze=False)
    fig.suptitle(f"{row['model_group']}: state covariance matrices", fontsize=14)
    vmax = np.nanmax(np.abs(covariances))
    for state_id in range(n_states):
        axis = axes[state_id // ncols][state_id % ncols]
        image = axis.imshow(covariances[state_id], cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        axis.set_title(f"State {state_id}")
        axis.set_xticks(range(len(features)), features, rotation=90, fontsize=8)
        axis.set_yticks(range(len(features)), features, fontsize=8)
        fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    for state_id in range(n_states, nrows * ncols):
        axes[state_id // ncols][state_id % ncols].set_axis_off()
    plt.show()

## Экспорт таблиц

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "analysis" / "hmm_grid_search" / "outputs" / f"refined_full_run_id={run_id}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

best_by_bic.to_csv(OUTPUT_DIR / "best_by_bic.csv", index=False)
best_by_aic.to_csv(OUTPUT_DIR / "best_by_aic.csv", index=False)
best_by_ll.to_csv(OUTPUT_DIR / "best_by_loglik.csv", index=False)
hyper_summary.to_csv(OUTPUT_DIR / "hyperparameter_summary.csv", index=False)
duration_stats_table.to_csv(OUTPUT_DIR / "best_bic_duration_stats.csv", index=False)
state_means_table.to_csv(OUTPUT_DIR / "best_bic_state_means.csv", index=False)
failed.to_csv(OUTPUT_DIR / "failed_models.csv", index=False)

print(f"Saved tables to: {OUTPUT_DIR}")